# Why rainbow colour maps mislead

**Why rainbow (jet) colour maps mislead -- measured, not asserted.**

A colour map turns numbers into colours. For that to be honest, equal steps in the DATA must look like equal steps in COLOUR. Rainbow maps fail this badly: they have bright bands that invent boundaries, and dark stretches that hide real differences.

**What it shows:**

- the same data in jet and in viridis -- jet grows features that are not there
- lightness plotted along each colour map, which is where jet's problem is
- a greyscale test: does the map still work in black and white?

---

*Chapter:* `color` — palettes, colour blindness, and why rainbow lies  
*Run the cells in order.* Every figure is also written to `viz/output/color/`, which is what the Streamlit gallery (`viz/project/gallery.py`) reads.


## Setup

These lines are how every notebook in the folder finds `vizkit.py`, which holds the save helpers and the seeded sample data. The data is seeded on purpose: your figures should come out identical to everyone else's.

`save()` writes each figure into `viz/output/` **and** leaves it on screen here. The trailing `;` on those calls only stops the notebook echoing the path it returns.


In [ ]:
%matplotlib inline

# A notebook has no __file__, so find viz/ by walking up from this
# notebook's own folder until vizkit.py turns up.
import sys
from pathlib import Path

VIZ = next(p for p in [Path.cwd(), *Path.cwd().parents]
           if (p / "vizkit.py").exists())
sys.path.insert(0, str(VIZ))

import matplotlib.pyplot as plt
import numpy as np

from vizkit import save

# Where save() files this lesson's output: viz/output/color/
LESSON = "color/rainbow"


## Measuring lightness

`lightness()` measures perceived brightness along a colour map using the standard sRGB luminance weights. Everything below is an argument about that one curve.


In [ ]:
def lightness(cmap_name, n=256):
    """Perceived lightness along a colour map (0 = black, 100 = white).

    Uses the standard luminance weights for sRGB. A good sequential map rises
    steadily; a rainbow map wanders up and down.
    """
    colours = plt.get_cmap(cmap_name)(np.linspace(0, 1, n))[:, :3]
    return 100 * (0.2126 * colours[:, 0] + 0.7152 * colours[:, 1]
                  + 0.0722 * colours[:, 2])


## 1. The same smooth data, two colour maps

The underlying data is a perfectly straight ramp. Any band, edge or feature you can see in the jet strip was invented by the colour map.


In [ ]:
# A perfectly smooth ramp. Any structure you see is the colour map's doing.
x = np.linspace(0, 1, 400)
smooth = np.tile(x, (60, 1))

fig, axes = plt.subplots(2, 1, figsize=(10, 3.4))
for ax, cmap in zip(axes, ["jet", "viridis"]):
    ax.imshow(smooth, cmap=cmap, aspect="auto")
    ax.set_title(f"{cmap}: a perfectly linear ramp", fontsize=10, loc="left")
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("The data is a straight line. jet invents bands in it.", fontsize=12)
fig.tight_layout()
save(fig, LESSON, "linear-ramp");


## 2. The lightness curve, which is the actual problem

Here is the mechanism. A good sequential map's lightness climbs steadily, so equal data steps look like equal colour steps. jet's goes up and down, and every crossing of zero on the right-hand panel is a place where a bigger number looks darker.


In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.6))

for cmap in ["jet", "viridis", "Greys"]:
    left.plot(np.linspace(0, 1, 256), lightness(cmap), label=cmap, lw=2)
left.set(xlabel="position in the colour map", ylabel="perceived lightness",
         title="Lightness should climb steadily")
left.legend()

jet_l = lightness("jet")
viridis_l = lightness("viridis")
right.plot(np.linspace(0, 1, 255), np.diff(jet_l), label="jet", lw=1.5)
right.plot(np.linspace(0, 1, 255), np.diff(viridis_l), label="viridis", lw=1.5)
right.axhline(0, color="black", lw=0.8)
right.set(xlabel="position", ylabel="change in lightness",
          title="jet goes UP and DOWN -- crossing zero invents edges")
right.legend()

fig.tight_layout()
save(fig, LESSON, "lightness");


## 3. Real data, and the greyscale test

The greyscale test is the cheapest check there is: strip the hue and see whether the picture still works. jet's rings vanish and its middle turns to mush, which is also what a photocopier does to it.


In [ ]:
yy, xx = np.mgrid[-3:3:200j, -3:3:200j]
field = np.exp(-(xx**2 + yy**2) / 4) + 0.35 * np.sin(3 * xx) * np.cos(3 * yy)

fig, axes = plt.subplots(2, 2, figsize=(9, 7))

axes[0, 0].imshow(field, cmap="jet");     axes[0, 0].set_title("jet")
axes[0, 1].imshow(field, cmap="viridis"); axes[0, 1].set_title("viridis")

# The greyscale test: convert each colour map's lightness only.
for ax, cmap in [(axes[1, 0], "jet"), (axes[1, 1], "viridis")]:
    colours = plt.get_cmap(cmap)(plt.Normalize()(field))[:, :, :3]
    grey = (0.2126 * colours[..., 0] + 0.7152 * colours[..., 1]
            + 0.0722 * colours[..., 2])
    ax.imshow(grey, cmap="gray")
    ax.set_title(f"{cmap}, printed in greyscale")

for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle("Greyscale test: jet's rings vanish and its middle turns to mush",
             fontsize=12)
fig.tight_layout()
save(fig, LESSON, "greyscale-test")

drops = int((np.diff(jet_l) < 0).sum())
print(f"""
  Measured on this machine:
    jet's lightness DECREASES at {drops} of 255 steps -- every one of those is
    a place where a bigger number looks darker than a smaller one.
    viridis decreases at {int((np.diff(viridis_l) < 0).sum())} steps.

  Use viridis / magma / cividis for continuous data.
  Keep jet for pretty pictures where nobody has to read a value.
""")


## Try it yourself

Edit the cells above and re-run them — that is what the notebook is for.

1. Add `turbo` (Google's rainbow replacement) to section 2's lightness plot. Is it monotonic? Does that make it safe?
2. Run the greyscale test on `coolwarm`. Why does a *diverging* map fail it, and why is that not automatically a bug?
3. Count the lightness reversals for every map in `plt.colormaps()[:40]` and print the worst five.


In [ ]:
# your turn


---

**Previous:** [`color/colorblind`](colorblind.ipynb)  
**Next:** [`annotation/direct_labels`](../annotation/direct_labels.ipynb)
